# Deforestation Monitor · U-Net forest segmentation (Kaggle / Colab)

Trains `smp.Unet("resnet34")` on 7-channel Sentinel-2 tiles (B02, B03, B04, B08, B11, B12, NDVI) with ESA WorldCover 2021 forest labels, then exports ONNX and scores the held-out region.

**Before running**
* Kaggle: *Settings → Accelerator → GPU (T4 or P100)* and *Internet → On* (needs a phone-verified account). Colab: *Runtime → Change runtime type → T4 GPU*.
* Upload the tiles once from your laptop: `python -m training.upload_tiles --repo <you>/deforestation-monitor-tiles`, then set `HF_REPO` below. If the dataset repo is private, add an `HF_TOKEN` secret.

**Outputs** (in `outputs/`): `unet_r34.onnx`, `norm_stats.json`, `metrics.json`, `history.json`, `best.pt`. Copy the first two into `models/` and `metrics.json` into `web/public/data/` of the repo.

In [ ]:
# --- Settings ----------------------------------------------------------------------
HF_REPO = "<your-hf-username>/deforestation-monitor-tiles"
EPOCHS = 30
BATCH_SIZE = 16
NUM_WORKERS = 2

In [ ]:
%pip install -q "segmentation-models-pytorch>=0.3.4" "onnx>=1.16" \
    "onnxruntime>=1.18" "huggingface_hub>=0.24"

In [ ]:
import json
import os
import sys
from pathlib import Path

import torch
from huggingface_hub import snapshot_download

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
DATA = WORK / "data"
OUT = WORK / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

token = os.environ.get("HF_TOKEN")
try:  # Kaggle secrets, if configured
    from kaggle_secrets import UserSecretsClient

    token = token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
snapshot_download(HF_REPO, repo_type="dataset", local_dir=DATA, token=token, max_workers=16)
sys.path.insert(0, str(DATA / "code"))

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, slow)")
print("tiles:", len((DATA / "manifest.csv").read_text().splitlines()) - 1)

In [ ]:
from training.train import TrainConfig, check_parity, export_onnx, fit

cfg = TrainConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
model, history = fit(DATA / "manifest.csv", DATA / "norm_stats.json", OUT, cfg)

In [ ]:
import matplotlib.pyplot as plt

ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ep, [h["train_loss"] for h in history], label="train")
ax[0].plot(ep, [h["val_loss"] for h in history], label="val")
ax[0].set_title("Dice + BCE loss")
ax[0].legend()
ax[1].plot(ep, [h["val_iou"] for h in history], label="val IoU")
ax[1].plot(ep, [h["val_f1"] for h in history], label="val F1")
ax[1].axhline(0.85, ls="--", c="grey", lw=0.8)
ax[1].set_title("Validation (Western Ghats)")
ax[1].legend()
plt.show()

## Export to ONNX and check parity with PyTorch (max |diff| ≤ 1e-3)

In [ ]:
import numpy as np

from training.dataset import TileDataset

onnx_path = export_onnx(model, OUT / "unet_r34.onnx", cfg)
val = TileDataset(DATA / "manifest.csv", ["val"], DATA / "norm_stats.json")
x = np.stack([val[i][0] for i in range(8)])
diff = check_parity(model, onnx_path, x)
print(f"ONNX parity OK: max |diff| = {diff:.2e};  size = {onnx_path.stat().st_size / 1e6:.1f} MB")

## Score the held-out region with the ONNX model

In [ ]:
import datetime as dt
import shutil

from training.evaluate import build_metrics, make_session, region_counts

per_region = region_counts(
    make_session(onnx_path), DATA / "manifest.csv", DATA / "norm_stats.json", ["test", "val"]
)
metrics = build_metrics(
    per_region,
    threshold=cfg.threshold,
    label_source="ESA WorldCover 2021 (class 10)",
    model=onnx_path.name,
    generated=dt.date.today().isoformat(),
)
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n")
shutil.copy2(DATA / "norm_stats.json", OUT / "norm_stats.json")

print(
    f"Held-out (test) IoU {metrics['iou']:.3f}  F1 {metrics['f1']:.3f}  "
    f"precision {metrics['precision']:.3f}  recall {metrics['recall']:.3f}  (target IoU > 0.85)"
)
for name, r in metrics["per_region"].items():
    print(
        f"  {name:14s} {r['split']:5s} tiles={r['n_tiles']:4d}  "
        f"IoU {r['iou']:.3f}  F1 {r['f1']:.3f}"
    )